# 🎬 MAX QUALITY Image-to-Video Generator
### CogVideoX-5B + RIFE 60fps + Real-ESRGAN 4K — ฟรี บน Google Colab

**Pipeline:**
1. **CogVideoX-5B** — generate วิดีโอคุณภาพสูง (ดีกว่า SVD มาก)
2. **RIFE** — เพิ่ม FPS จาก 8 → 60 (smooth มาก)
3. **Real-ESRGAN** — upscale เป็น 4K

⚠️ **ก่อนเริ่ม:** Runtime → Change runtime type → **T4 GPU**

## 📦 ขั้นที่ 1 — ติดตั้ง Libraries (รอ 5 นาที)

In [ ]:
import subprocess, sys
print('🔧 ติดตั้ง libraries...')
pkgs = [
    'diffusers>=0.30.0', 'transformers>=4.40.0', 'accelerate>=0.30.0',
    'torch>=2.2.0', 'torchvision', 'Pillow', 'imageio', 'imageio-ffmpeg',
    'opencv-python-headless', 'einops', 'safetensors', 'sentencepiece',
    'basicsr', 'facexlib', 'gfpgan', 'realesrgan'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('✅ ติดตั้งเสร็จ!')

## 🔍 ขั้นที่ 2 — ตรวจสอบ GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} | VRAM: {vram:.1f} GB')
    if vram < 14:
        print('⚠️  VRAM ต่ำกว่า 14GB — จะใช้ memory optimization อัตโนมัติ')
else:
    print('❌ ไม่พบ GPU! ไปที่ Runtime → Change runtime type → T4 GPU')

## 🤖 ขั้นที่ 3 — โหลด CogVideoX-5B Model (รอ 10-15 นาที ครั้งแรก)

In [ ]:
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
import torch

print('📥 โหลด CogVideoX-5B...')
print('   ครั้งแรกใช้เวลา 10-15 นาที เพราะ model ใหญ่ 20GB')

pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    'THUDM/CogVideoX-5b-I2V',
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print('✅ โหลด Model เสร็จแล้ว!')

## 🖼️ ขั้นที่ 4 — อัปโหลดรูป

In [ ]:
from google.colab import files
from PIL import Image
from IPython.display import display
import io

print('📁 เลือกรูปที่ต้องการแปลงเป็นวิดีโอ:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')
image = image.resize((720, 480))
print(f'✅ โหลดรูป: {filename}')
display(image.resize((480, 320)))

## ⚙️ ขั้นที่ 5 — ตั้งค่า

| MOTION | ผล |
|---|---|
| ต่ำ (0-30) | กล้องนิ่ง เหมาะกับห้อง ฝนตก เทียน |
| กลาง (40-60) | เคลื่อนไหวเบา ลม ใบไม้ |
| สูง (70-100) | เคลื่อนไหวเยอะ คลื่น พายุ |

In [ ]:
# ============================================================
# 🎛️ ตั้งค่าตรงนี้!
# ============================================================

# คำอธิบายวิดีโอ (ภาษาอังกฤษ)
# ยิ่งละเอียด ยิ่งดี!
PROMPT = """A cozy Japanese bedroom at night, heavy rain falling outside the window,
raindrops streaming down the glass, dim warm lamp light, absolutely static camera,
no camera movement, peaceful and calming atmosphere, cinematic quality"""

# สิ่งที่ไม่ต้องการ
NEGATIVE_PROMPT = "blur, distortion, ugly, camera shake, fast motion, low quality"

# จำนวน frames: 17 (เร็ว) | 33 (ยาวขึ้น)
NUM_FRAMES = 33

# Guidance scale: ยิ่งสูง = ยึด prompt มากขึ้น (แนะนำ 6-8)
GUIDANCE_SCALE = 7.0

# Steps: ยิ่งสูง = ละเอียดขึ้น แต่ช้ากว่า (แนะนำ 30-50)
NUM_STEPS = 40

# Seed: -1=สุ่ม
SEED = -1

# ============================================================
print('✅ ตั้งค่าเสร็จ!')
print(f'   Prompt  : {PROMPT[:80]}...')
print(f'   Frames  : {NUM_FRAMES} | Steps: {NUM_STEPS} | Guidance: {GUIDANCE_SCALE}')

## 🚀 ขั้นที่ 6 — Generate วิดีโอ (รอ 10-20 นาที)

In [ ]:
import torch, time, os
from diffusers.utils import export_to_video

print('🎬 เริ่ม Generate ด้วย CogVideoX-5B...')
print('   อย่าปิด browser!')

generator = None if SEED == -1 else torch.Generator().manual_seed(SEED)
start = time.time()

with torch.inference_mode():
    result = pipe(
        image=image,
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_frames=NUM_FRAMES,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
        generator=generator,
    )

frames = result.frames[0]
elapsed = time.time() - start
print(f'✅ Generate เสร็จ! ใช้เวลา {elapsed/60:.1f} นาที')

raw_path = '/content/raw_video.mp4'
export_to_video(frames, raw_path, fps=8)
print(f'💾 บันทึก raw video: {raw_path}')

## 🚀 ขั้นที่ 7 — RIFE Frame Interpolation (8fps → 60fps)

In [ ]:
import subprocess, os

# โหลด RIFE
if not os.path.exists('/content/ECCV2022-RIFE'):
    print('📥 โหลด RIFE...')
    subprocess.run(['git', 'clone', 'https://github.com/megvii-research/ECCV2022-RIFE', '/content/ECCV2022-RIFE'], check=True)
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    os.makedirs('/content/ECCV2022-RIFE/train_log', exist_ok=True)
    subprocess.run(['gdown', '--fuzzy',
        'https://drive.google.com/file/d/1APIzVeI-4ZZCEuIRE1m6WYfSCaOsi_7_/view',
        '-O', '/content/ECCV2022-RIFE/train_log/IFNet_trained.pkl'], check=True)
    print('✅ โหลด RIFE เสร็จ!')

# แยก frames จาก raw video
frames_dir = '/content/rife_input'
os.makedirs(frames_dir, exist_ok=True)
subprocess.run([
    'ffmpeg', '-y', '-i', raw_path,
    '-qscale:v', '2', f'{frames_dir}/frame%04d.png'
], check=True, capture_output=True)

frame_count = len([f for f in os.listdir(frames_dir) if f.endswith('.png')])
print(f'✅ แยกได้ {frame_count} frames')

# รัน RIFE interpolation (8fps → 64fps ประมาณ)
rife_out = '/content/rife_output'
os.makedirs(rife_out, exist_ok=True)

print('🎞️  กำลัง interpolate frames (เพิ่ม fps)...')
subprocess.run([
    'python3', '/content/ECCV2022-RIFE/inference_img.py',
    '--img', frames_dir,
    '--exp', '3',  # 2^3 = 8x interpolation
    '--output', rife_out
], check=True)

# รวมเป็น video 60fps
smooth_path = '/content/smooth_video.mp4'
subprocess.run([
    'ffmpeg', '-y',
    '-framerate', '60',
    '-pattern_type', 'glob', '-i', f'{rife_out}/*.png',
    '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p',
    smooth_path
], check=True, capture_output=True)

size_mb = os.path.getsize(smooth_path) / 1024 / 1024
print(f'✅ Smooth video (60fps) พร้อมแล้ว: {size_mb:.1f} MB')

## ✨ ขั้นที่ 8 — Real-ESRGAN Upscale 4K

In [ ]:
import subprocess, os

# โหลด Real-ESRGAN
if not os.path.exists('/content/Real-ESRGAN'):
    print('📥 โหลด Real-ESRGAN...')
    subprocess.run(['git', 'clone', 'https://github.com/xinntao/Real-ESRGAN', '/content/Real-ESRGAN'], check=True)
    subprocess.run(['pip', 'install', '-q', '-r', '/content/Real-ESRGAN/requirements.txt'], check=True)
    subprocess.run(['python', '/content/Real-ESRGAN/setup.py', 'develop'],
                   cwd='/content/Real-ESRGAN', check=True, capture_output=True)

# แยก frames จาก smooth video
esrgan_in = '/content/esrgan_input'
esrgan_out = '/content/esrgan_output'
os.makedirs(esrgan_in, exist_ok=True)
os.makedirs(esrgan_out, exist_ok=True)

print('🖼️  แยก frames สำหรับ upscale...')
subprocess.run([
    'ffmpeg', '-y', '-i', smooth_path,
    '-qscale:v', '1', f'{esrgan_in}/frame%06d.png'
], check=True, capture_output=True)

n = len([f for f in os.listdir(esrgan_in) if f.endswith('.png')])
print(f'   {n} frames — กำลัง upscale เป็น 4K (ใช้เวลาสักครู่)...')

# รัน ESRGAN upscale x4
subprocess.run([
    'python', '/content/Real-ESRGAN/inference_realesrgan.py',
    '-n', 'RealESRGAN_x4plus',
    '-i', esrgan_in,
    '-o', esrgan_out,
    '--outscale', '4',
    '--fp32'
], cwd='/content/Real-ESRGAN', check=True)

# รวมเป็น video 4K 60fps
final_path = '/content/FINAL_4K_60fps.mp4'
subprocess.run([
    'ffmpeg', '-y',
    '-framerate', '60',
    '-pattern_type', 'glob', '-i', f'{esrgan_out}/*.png',
    '-c:v', 'libx264', '-crf', '16', '-pix_fmt', 'yuv420p',
    '-vf', 'scale=3840:2160:flags=lanczos',
    final_path
], check=True, capture_output=True)

size_mb = os.path.getsize(final_path) / 1024 / 1024
print(f'✅ FINAL 4K 60fps video: {size_mb:.1f} MB')
print('   👇 รัน Cell ถัดไปเพื่อ Download')

## 📥 ขั้นที่ 9 — Preview และ Download

In [ ]:
from IPython.display import HTML, display
from google.colab import files
import base64, os

# Preview จาก smooth video (ไฟล์เล็กกว่า preview ง่ายกว่า)
with open(smooth_path, 'rb') as f:
    v = base64.b64encode(f.read()).decode()

display(HTML(f'''
<div style="text-align:center;padding:20px;background:#111;border-radius:12px">
  <h3 style="color:white">🎬 Preview (60fps smooth)</h3>
  <video width="640" controls loop autoplay muted style="border-radius:8px">
    <source src="data:video/mp4;base64,{v}" type="video/mp4">
  </video>
</div>
'''))

print('\n📥 Download ไฟล์ทั้งหมด:')
print(f'   1. Raw video (8fps)')
files.download(raw_path)
print(f'   2. Smooth video (60fps)')
files.download(smooth_path)
print(f'   3. FINAL 4K 60fps ⭐')
files.download(final_path)
print('✅ Download เสร็จ!')

## 💡 Tips

**Prompt ที่ดีสำหรับห้องนอนฝนตก:**
```
A cozy Japanese bedroom, heavy rain outside window, raindrops on glass,
warm lamp light, static camera, no movement, peaceful, cinematic 4K
```

**Generate ใหม่:** กลับไปแก้ Cell 5 แล้วรัน Cell 6-9 ใหม่ ไม่ต้องโหลด model ซ้ำ

**ถ้า RAM เต็ม:** Runtime → Restart session แล้วรันจาก Cell 3 ใหม่